# LoRA Final Project (Notebook-First, From Scratch)

This notebook is fully runnable in Colab and keeps the core workflow inside notebook cells.
- LoRA implementation from scratch (no PEFT LoRA call)
- Training + evaluation in notebook
- Figure generation in notebook
- Auto-export artifacts

Hyperparameters default to paper-aligned settings where applicable:
- GLUE SST-2/MRPC: from official LoRA NLU scripts
- GPT-2 LoRA defaults: from official LoRA NLG setup


In [ ]:
import os, sys, json, math, random, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader

IN_COLAB = 'COLAB_GPU' in os.environ
print('Running in Colab:', IN_COLAB)
if IN_COLAB:
    !nvidia-smi || true


In [ ]:
!python -m pip install --upgrade pip -q
!pip install -q torch transformers datasets tqdm matplotlib pandas


In [ ]:
# Mount Google Drive (recommended for reliable artifact persistence)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive mounted at /content/drive')
else:
    print('Not in Colab; skipping Drive mount.')


In [ ]:
REPO_URL = "https://github.com/rohan-shankar/cs4782-lora-implementation.git"
PROJECT_DIR = Path('/content/deep-learning-final-project') if IN_COLAB else Path.cwd()
if IN_COLAB and not PROJECT_DIR.exists():
    !git clone "$REPO_URL" "$PROJECT_DIR"
os.chdir(PROJECT_DIR)
print('Project dir:', PROJECT_DIR)


In [ ]:
RESULTS_ROOT = PROJECT_DIR / 'results'
EXPORT_ROOT = RESULTS_ROOT / 'exports'
for p in [RESULTS_ROOT, EXPORT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)


In [ ]:
PAPER_GLUE = {
    # From LoRA official scripts (roberta_base_sst2.sh, roberta_base_mrpc.sh)
    'sst2': {
        'max_seq_length': 512,
        'per_device_train_batch_size': 16,
        'per_device_eval_batch_size': 32,
        'learning_rate': 5e-4,
        'num_train_epochs': 60,
        'warmup_ratio': 0.06,
        'weight_decay': 0.1,
        'lora_r': 8,
        'lora_alpha': 16,
        'seed': 0,
    },
    'mrpc': {
        'max_seq_length': 512,
        'per_device_train_batch_size': 16,
        'per_device_eval_batch_size': 32,
        'learning_rate': 4e-4,
        'num_train_epochs': 30,
        'warmup_ratio': 0.06,
        'weight_decay': 0.1,
        'lora_r': 8,
        'lora_alpha': 16,
        'seed': 0,
    },
}

PAPER_GPT2 = {
    # From LoRA official GPT-2 setup (NLG README / gpt2_ft args)
    'seq_len': 512,
    'train_batch_size': 8,
    'eval_batch_size': 4,
    'learning_rate': 2e-4,
    'weight_decay': 0.01,
    'warmup_steps': 500,
    'num_train_epochs': 5,
    'adam_beta2': 0.999,
    'lora_r': 4,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'seed': 110,
}

print('Loaded paper-aligned hyperparameter presets.')


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2)


def zip_paths(out_zip: Path, paths):
    out_zip.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in paths:
            p = Path(p)
            if not p.exists():
                continue
            if p.is_file():
                zf.write(p, arcname=p.name)
            else:
                for f in p.rglob('*'):
                    if f.is_file():
                        zf.write(f, arcname=str(f.relative_to(PROJECT_DIR)))
    return out_zip


def auto_export(paths, zip_name='artifacts.zip'):
    out_zip = EXPORT_ROOT / zip_name
    zip_paths(out_zip, paths)
    print('Created:', out_zip)
    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        if drive_root.exists():
            drive_dir = drive_root / 'lora_project_exports'
            drive_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(out_zip, drive_dir / out_zip.name)
            print('Copied to Drive:', drive_dir / out_zip.name)
        try:
            from google.colab import files
            files.download(str(out_zip))
            print('Download triggered.')
        except Exception as e:
            print('Auto-download skipped:', e)
    return out_zip


In [ ]:
from transformers.pytorch_utils import Conv1D

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        assert rank > 0
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False) if base.bias is not None else None
        self.lora_A = nn.Parameter(torch.empty(rank, base.in_features))
        self.lora_B = nn.Parameter(torch.empty(base.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias) + self.scaling * F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B)

class LoRAConv1D(nn.Module):
    def __init__(self, base: Conv1D, rank: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        assert rank > 0
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        in_features, out_features = base.weight.shape
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        return F.linear(x, self.weight.transpose(0, 1), self.bias) + self.scaling * F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B)


def _get_parent(root, name):
    parts = name.split('.')
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]


def apply_lora(model, target_suffixes, rank, alpha, dropout=0.0):
    replaced = []
    for name, module in list(model.named_modules()):
        if not any(name.endswith(s) for s in target_suffixes):
            continue
        parent, child = _get_parent(model, name)
        if isinstance(module, nn.Linear):
            repl = LoRALinear(module, rank, alpha, dropout)
        elif isinstance(module, Conv1D):
            repl = LoRAConv1D(module, rank, alpha, dropout)
        else:
            continue
        setattr(parent, child, repl)
        replaced.append(name)
    if not replaced:
        raise ValueError(f'No modules replaced for suffixes={target_suffixes}')
    return replaced


def freeze_for_lora(model, extra_trainable=()):
    for n, p in model.named_parameters():
        p.requires_grad = False
        if 'lora_A' in n or 'lora_B' in n or any(k in n for k in extra_trainable):
            p.requires_grad = True


In [ ]:
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
    DataCollatorWithPadding, DataCollatorForLanguageModeling, get_linear_schedule_with_warmup,
)

TASK_TO_KEYS = {
    'cola': ('sentence', None),
    'sst2': ('sentence', None),
    'mrpc': ('sentence1', 'sentence2'),
    'qqp': ('question1', 'question2'),
}

def f1_binary(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1)); fp = np.sum((y_true == 0) & (y_pred == 1)); fn = np.sum((y_true == 1) & (y_pred == 0))
    p = tp / (tp + fp + 1e-12); r = tp / (tp + fn + 1e-12)
    return float(2 * p * r / (p + r + 1e-12))


def run_glue(task='sst2', mode='full', rank=None, alpha=None, seed=None, epochs=None, max_steps=-1, out_root='results/notebook_glue'):
    cfg = PAPER_GLUE[task]
    rank = cfg['lora_r'] if rank is None else rank
    alpha = cfg['lora_alpha'] if alpha is None else alpha
    seed = cfg['seed'] if seed is None else seed
    epochs = cfg['num_train_epochs'] if epochs is None else epochs

    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()

    out_dir = PROJECT_DIR / out_root / f"{task}_{mode}_r{rank}_seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)

    s1, s2 = TASK_TO_KEYS[task]
    ds = load_dataset('glue', task)
    tok = AutoTokenizer.from_pretrained('bert-base-uncased')

    def preprocess(ex):
        if s2 is None: return tok(ex[s1], truncation=True, max_length=cfg['max_seq_length'])
        return tok(ex[s1], ex[s2], truncation=True, max_length=cfg['max_seq_length'])

    enc = ds.map(preprocess, batched=True)
    enc = enc.rename_column('label', 'labels')
    enc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

    train_loader = DataLoader(enc['train'], batch_size=cfg['per_device_train_batch_size'], shuffle=True, collate_fn=DataCollatorWithPadding(tok))
    val_loader = DataLoader(enc['validation'], batch_size=cfg['per_device_eval_batch_size'], shuffle=False, collate_fn=DataCollatorWithPadding(tok))

    model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=enc['train'].features['labels'].num_classes)
    replaced = []
    if mode == 'lora':
        replaced = apply_lora(model, target_suffixes=('query','value'), rank=rank, alpha=alpha, dropout=0.0)
        freeze_for_lora(model, extra_trainable=('classifier',))

    model.to(device)
    total_params, trainable_params = count_params(model)
    optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])
    total_train_steps = max_steps if max_steps > 0 else epochs * len(train_loader)
    sched = get_linear_schedule_with_warmup(optim, int(cfg['warmup_ratio'] * total_train_steps), total_train_steps)

    step, history, best_acc, best = 0, [], -1.0, {}
    for ep in range(epochs):
        model.train(); losses=[]
        for batch in tqdm(train_loader, desc=f'{task}-{mode} train ep{ep+1}'):
            batch = {k:v.to(device) for k,v in batch.items()}
            loss = model(**batch).loss
            loss.backward(); optim.step(); sched.step(); optim.zero_grad(set_to_none=True)
            losses.append(float(loss.item())); step += 1
            if step >= total_train_steps: break

        model.eval(); all_logits=[]; all_labels=[]
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'{task}-{mode} eval'):
                labels = batch['labels'].numpy()
                batch = {k:v.to(device) for k,v in batch.items()}
                all_logits.append(model(**batch).logits.detach().cpu().numpy()); all_labels.append(labels)

        logits = np.concatenate(all_logits, axis=0); labels = np.concatenate(all_labels, axis=0); preds = logits.argmax(axis=-1)
        rec = {'epoch': ep+1, 'train_loss': float(np.mean(losses)), 'accuracy': float((preds==labels).mean())}
        if task in ('mrpc','qqp'): rec['f1'] = f1_binary(labels, preds)
        history.append(rec)

        if rec['accuracy'] > best_acc:
            best_acc = rec['accuracy']; best = dict(rec); torch.save(model.state_dict(), out_dir / 'best_model.pt')
        if step >= total_train_steps: break

    peak_mem = float(torch.cuda.max_memory_allocated()/(1024**3)) if device.type=='cuda' else 0.0
    ckpt = out_dir / 'best_model.pt'; ckpt_mb = float(ckpt.stat().st_size/(1024**2)) if ckpt.exists() else 0.0
    summary = {
        'task_name': task, 'mode': mode,
        'paper_config': cfg,
        'lora_rank': rank if mode=='lora' else None, 'lora_alpha': alpha if mode=='lora' else None,
        'replaced_modules': replaced,
        'total_params': total_params, 'trainable_params': trainable_params, 'trainable_ratio': trainable_params/total_params,
        'peak_gpu_memory_gb': peak_mem, 'checkpoint_size_mb': ckpt_mb,
        'best_metrics': best, 'history': history,
    }
    save_json(out_dir / 'summary.json', summary)
    return summary, out_dir


def run_wikitext(mode='full', rank=None, alpha=None, seed=None, epochs=None, max_steps=-1, out_root='results/notebook_wikitext2'):
    cfg = PAPER_GPT2
    rank = cfg['lora_r'] if rank is None else rank
    alpha = cfg['lora_alpha'] if alpha is None else alpha
    seed = cfg['seed'] if seed is None else seed
    epochs = cfg['num_train_epochs'] if epochs is None else epochs

    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()

    out_dir = PROJECT_DIR / out_root / f'{mode}_r{rank}_seed{seed}'
    out_dir.mkdir(parents=True, exist_ok=True)

    ds = load_dataset('wikitext', 'wikitext-2-raw-v1')
    tok = AutoTokenizer.from_pretrained('gpt2')
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    def tokenize(batch): return tok(batch['text'])
    tokds = ds.map(tokenize, batched=True, remove_columns=['text'])

    block = cfg['seq_len']
    def group(examples):
        concat = {k: sum(examples[k], []) for k in examples.keys()}
        L = (len(concat['input_ids']) // block) * block
        out = {k: [t[i:i+block] for i in range(0, L, block)] for k,t in concat.items()}
        out['labels'] = [x.copy() for x in out['input_ids']]
        return out

    lm = tokds.map(group, batched=True)
    lm.set_format(type='torch')
    collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
    tr = DataLoader(lm['train'], batch_size=cfg['train_batch_size'], shuffle=True, collate_fn=collator)
    va = DataLoader(lm['validation'], batch_size=cfg['eval_batch_size'], shuffle=False, collate_fn=collator)

    model = AutoModelForCausalLM.from_pretrained('gpt2')
    replaced=[]
    if mode=='lora':
        replaced = apply_lora(model, target_suffixes=('c_attn',), rank=rank, alpha=alpha, dropout=cfg['lora_dropout'])
        freeze_for_lora(model)

    model.to(device)
    total_params, trainable_params = count_params(model)
    optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'], betas=(0.9, cfg['adam_beta2']))
    total_train_steps = max_steps if max_steps > 0 else epochs * len(tr)
    warmup_steps = min(cfg['warmup_steps'], total_train_steps // 2) if total_train_steps > 1 else 0
    sched = get_linear_schedule_with_warmup(optim, warmup_steps, total_train_steps)

    step, history, best_ppl, best = 0, [], float('inf'), {}
    for ep in range(epochs):
        model.train(); train_losses=[]
        for batch in tqdm(tr, desc=f'wikitext-{mode} train ep{ep+1}'):
            batch = {k:v.to(device) for k,v in batch.items()}
            loss = model(**batch).loss
            loss.backward(); optim.step(); sched.step(); optim.zero_grad(set_to_none=True)
            train_losses.append(float(loss.item())); step += 1
            if step >= total_train_steps: break

        model.eval(); eval_losses=[]
        with torch.no_grad():
            for batch in tqdm(va, desc=f'wikitext-{mode} eval'):
                batch = {k:v.to(device) for k,v in batch.items()}
                eval_losses.append(float(model(**batch).loss.item()))

        eval_loss = float(np.mean(eval_losses)); ppl = float(math.exp(eval_loss))
        rec = {'epoch': ep+1, 'train_loss': float(np.mean(train_losses)), 'eval_loss': eval_loss, 'perplexity': ppl}
        history.append(rec)
        if ppl < best_ppl:
            best_ppl = ppl; best = dict(rec); torch.save(model.state_dict(), out_dir / 'best_model.pt')
        if step >= total_train_steps: break

    peak_mem = float(torch.cuda.max_memory_allocated()/(1024**3)) if device.type=='cuda' else 0.0
    ckpt = out_dir / 'best_model.pt'; ckpt_mb = float(ckpt.stat().st_size/(1024**2)) if ckpt.exists() else 0.0
    summary = {
        'dataset': 'wikitext-2-raw-v1', 'mode': mode,
        'paper_config': cfg,
        'lora_rank': rank if mode=='lora' else None, 'lora_alpha': alpha if mode=='lora' else None,
        'replaced_modules': replaced,
        'total_params': total_params, 'trainable_params': trainable_params, 'trainable_ratio': trainable_params/total_params,
        'peak_gpu_memory_gb': peak_mem, 'checkpoint_size_mb': ckpt_mb,
        'best_metrics': best, 'history': history,
    }
    save_json(out_dir / 'summary.json', summary)
    return summary, out_dir


In [ ]:
def load_summaries(root: Path):
    rows = []
    for p in root.rglob('summary.json'):
        with open(p, 'r', encoding='utf-8') as f: obj = json.load(f)
        best = obj.get('best_metrics', {})
        for k, v in best.items(): obj[f'best_{k}'] = v
        obj['path'] = str(p)
        rows.append(obj)
    if not rows: raise RuntimeError('No summary files found')
    return pd.DataFrame(rows)


def generate_main_figures(task='sst2', root=RESULTS_ROOT, outdir=RESULTS_ROOT / 'figures'):
    outdir.mkdir(parents=True, exist_ok=True)
    df = load_summaries(root); df.to_csv(outdir / 'all_summaries.csv', index=False)
    metric_col = 'best_accuracy' if task not in ('mrpc','qqp','cola') else ('best_f1' if task in ('mrpc','qqp') else 'best_mcc')
    g = df[df.get('task_name', pd.Series(dtype=str)) == task].copy()
    if g.empty: raise RuntimeError(f'No rows for task={task}')
    g['method'] = g.apply(lambda r: 'Full FT' if r.get('mode')=='full' else f"LoRA r={int(r.get('lora_rank'))}", axis=1)

    a = g[['method', metric_col]].dropna().copy().sort_values('method')
    plt.figure(figsize=(8.8,4.8)); bars = plt.bar(a['method'], a[metric_col], color='#2f6b9a')
    plt.title(f'Performance vs Method ({task.upper()})'); plt.ylabel(metric_col.replace('best_','').upper()); plt.xticks(rotation=22, ha='right')
    for b in bars:
        h = b.get_height(); plt.text(b.get_x()+b.get_width()/2, h, f'{h:.3f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout(); plt.savefig(outdir / f'01_performance_vs_method_{task}.png', dpi=220); plt.close()

    b = g[['method', metric_col, 'trainable_params']].dropna().copy()
    plt.figure(figsize=(7.2,4.8))
    for _, r in b.iterrows():
        plt.scatter(r['trainable_params'], r[metric_col], s=65); plt.annotate(r['method'], (r['trainable_params'], r[metric_col]), fontsize=8, xytext=(4,4), textcoords='offset points')
    plt.xscale('log'); plt.xlabel('Trainable Parameters (log)'); plt.ylabel(metric_col.replace('best_','').upper())
    plt.title(f'Performance vs Trainable Params ({task.upper()})')
    plt.tight_layout(); plt.savefig(outdir / f'02_performance_vs_trainable_params_{task}.png', dpi=220); plt.close()

    c = g[g['mode']=='lora'][['lora_rank', metric_col]].dropna().sort_values('lora_rank')
    if not c.empty:
        plt.figure(figsize=(7.2,4.8)); plt.plot(c['lora_rank'], c[metric_col], marker='o', color='#1f8a70', linewidth=2)
        plt.xlabel('LoRA Rank r'); plt.ylabel(metric_col.replace('best_','').upper()); plt.title(f'Performance vs LoRA Rank ({task.upper()})'); plt.grid(alpha=0.25)
        plt.tight_layout(); plt.savefig(outdir / f'03_performance_vs_lora_rank_{task}.png', dpi=220); plt.close()

    resource = 'peak_gpu_memory_gb' if g['peak_gpu_memory_gb'].fillna(0).max() > 0 else 'checkpoint_size_mb'
    d = g[['method', resource]].dropna().copy().sort_values('method')
    if not d.empty:
        plt.figure(figsize=(8.8,4.8)); plt.bar(d['method'], d[resource], color='#8f3b76'); plt.xticks(rotation=22, ha='right')
        plt.ylabel('Peak GPU Memory (GB)' if resource=='peak_gpu_memory_gb' else 'Checkpoint Size (MB)')
        plt.title(f"{'Memory' if resource=='peak_gpu_memory_gb' else 'Storage'} Savings ({task.upper()})")
        plt.tight_layout(); plt.savefig(outdir / (f'04_memory_savings_{task}.png' if resource=='peak_gpu_memory_gb' else f'04_storage_savings_{task}.png'), dpi=220); plt.close()

    curve_rows = []
    for _, row in g.iterrows():
        for h in (row.get('history') or []):
            curve_rows.append({'method': row['method'], 'epoch': h.get('epoch'), 'train_loss': h.get('train_loss')})
    if curve_rows:
        cr = pd.DataFrame(curve_rows).dropna()
        if not cr.empty:
            plt.figure(figsize=(7.2,4.8))
            for label, sub in cr.groupby('method'): plt.plot(sub['epoch'], sub['train_loss'], marker='o', label=label)
            plt.xlabel('Epoch'); plt.ylabel('Train Loss'); plt.title(f'Training Curves ({task.upper()})'); plt.legend(fontsize=8)
            plt.tight_layout(); plt.savefig(outdir / f'05_training_curves_{task}.png', dpi=220); plt.close()

    print('Figures saved to', outdir)
    return outdir


## Full Run (Paper-Aligned Hyperparameters)
This run can take several hours on Colab depending on GPU availability.


In [ ]:
full_out = []
for task in ['sst2', 'mrpc']:
    s, out = run_glue(task=task, mode='full', out_root='results/notebook_glue_full')
    full_out.append(out)
    for r in [1, 4, 8, 16]:
        s, out = run_glue(task=task, mode='lora', rank=r, out_root='results/notebook_glue_full')
        full_out.append(out)

s, out = run_wikitext(mode='full', out_root='results/notebook_wikitext2_full')
full_out.append(out)
for r in [1, 4, 8, 16]:
    s, out = run_wikitext(mode='lora', rank=r, out_root='results/notebook_wikitext2_full')
    full_out.append(out)

fig_sst2 = generate_main_figures(task='sst2', root=RESULTS_ROOT, outdir=RESULTS_ROOT / 'figures_full')
fig_mrpc = generate_main_figures(task='mrpc', root=RESULTS_ROOT, outdir=RESULTS_ROOT / 'figures_full')
auto_export([fig_sst2, fig_mrpc, PROJECT_DIR / 'results/notebook_glue_full', PROJECT_DIR / 'results/notebook_wikitext2_full'], 'full_outputs.zip')


In [ ]:
!find results/figures_full -maxdepth 1 -type f | sort


In [ ]:
# Force export (full) - use this if browser download was flaky in VSCode/Colab
from pathlib import Path
import shutil, zipfile

project = Path(PROJECT_DIR)
zip_path = project / 'full_outputs.zip'
targets = [
    project / 'results/figures_full',
    project / 'results/notebook_glue_full',
    project / 'results/notebook_wikitext2_full',
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for t in targets:
        if not t.exists():
            continue
        if t.is_file():
            zf.write(t, arcname=t.name)
        else:
            for f in t.rglob('*'):
                if f.is_file():
                    zf.write(f, arcname=str(f.relative_to(project)))

print('Created:', zip_path)
drive_root = Path('/content/drive/MyDrive')
if drive_root.exists():
    out_dir = drive_root / 'lora_project_exports'
    out_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, out_dir / zip_path.name)
    print('Copied to Drive:', out_dir / zip_path.name)

try:
    from google.colab import files
    files.download(str(zip_path))
    print('Download triggered')
except Exception as e:
    print('Download not triggered in this environment:', e)
